# COE Premium Dynamics in Singapore: Supply Mechanics, Auction Pressure, and Price Volatility

---

## Executive Summary

Singapore's Certificate of Entitlement system caps the vehicle population through a twice-monthly sealed-bid auction. Category A premiums have ranged from under S$30,000 to over S$100,000 within the same decade, despite the vehicle population stabilising near 320,000 units since 2016. This note investigates the structural drivers of that volatility using administrative data from the SingStat TableBuilder API (Feb 2002 – Feb 2026).

**Headline findings:**

1. Deregistrations Granger-cause quota allocations at a 6-month lag (p=0.003), consistent with LTA's administrative processing cycle. Supply is backward-looking by design.
2. Quota changes are the dominant short-run predictor of premium returns. A 1,000-unit increase in monthly quota is associated with a −2% log return in premium (p<0.001, HAC SE). The bid-to-quota ratio fails both Granger and HAC-robust OLS tests — not because demand pressure is irrelevant, but because the ratio is an equilibrium outcome that adjusts contemporaneously with price, not a forcing variable that precedes it.
3. Level correlations between supply variables and premiums are largely spurious — driven by shared trends rather than structural causation. Relationships collapse in first differences.
4. Formal break date testing (Chow, CUSUM, grid search) identifies a structural shift in the supply-premium relationship. The post-break regime is characterised by a 77% higher mean premium and half the monthly volatility of the growth phase.
5. Short-run premium returns are close to unpredictable from public administrative data alone. The ARIMAX model marginally outperforms a naive random walk on MAE but not RMSE over the 2021–2026 test period.
6. The 5-year outlook is structurally bearish: the 2016–2021 registration cohort reaches 10-year expiry between 2026 and 2031, mechanically increasing quota supply. Current premiums sit +2.4 standard deviations above the post-break mean. The macro environment — elevated oil prices, US tariff headwinds, slowing growth — adds asymmetric downside risk.

The COE market is a useful laboratory for administratively constrained asset pricing — analogous to spectrum auctions, fishing quotas, and carbon permits — where supply is institutionally lagged and demand shocks are absorbed entirely through price.

---

## 1. Institutional Background

### The COE System

Singapore controls its vehicle population through the Vehicle Quota System (VQS), introduced in 1990. To register a new vehicle, a buyer must first obtain a Certificate of Entitlement via a sealed-bid auction held twice monthly. COEs are valid for 10 years and are renewable at the Prevailing Quota Premium (PQP) — a 3-month moving average of recent clearing prices.

This analysis focuses on **Category A** (cars ≤1,600cc and ≤97kW), the largest and most liquid category, accounting for the majority of private car registrations.

### Stock Constraint vs. Flow Supply

The VQS imposes a stock constraint on the total vehicle population, but the monthly auction quota is a flow variable derived from prior-period deregistrations plus LTA discretionary adjustments. This creates a structural lag: deregistrations in month *t* feed into quota approximately 6–9 months later. Supply is backward-looking by design — it reflects what happened to the vehicle stock in the past, not what demand is doing today.

This asymmetry is the central economic feature of the COE market. In a standard competitive market, supply responds to price signals. In the VQS, supply is administratively determined and price-inelastic in the short run. The entire burden of market clearing falls on the demand side, which means price.

### Why Premiums Remain Volatile Under a Capped Stock

Even with a stable vehicle population, premiums are volatile for several reinforcing reasons:

- **Supply rigidity.** Quota is set by past deregistrations, not current demand. A demand surge cannot be met with more supply within the same auction cycle.
- **Auction mechanism.** The uniform-price sealed-bid format creates incentives to bid above reservation value. All winners pay the lowest winning bid, so overbidding is individually rational when the cost of losing the auction (waiting another fortnight, or longer) is high.
- **Inelastic demand at the margin.** COEs are a prerequisite good with no close substitute for buyers who need a car. Demand does not fall proportionally with price increases.
- **Cohort expiry.** Vehicles registered in the same year reach end-of-life simultaneously, generating lumpy and predictable deregistration waves — and hence lumpy quota supply — that do not align with demand cycles.
- **Expectations and forward bidding.** Bidders who anticipate tighter future supply bid more aggressively today, amplifying price spikes before the supply constraint even binds.

---

## 2. Research Questions and Hypotheses

The analysis is organised around four testable claims about the supply-premium relationship. These are stated before the data is examined and assessed against formal test results in Section 10.

**H1 — Deregistration-Quota Lag:** Deregistrations Granger-cause quota allocations at a lag of approximately 6–9 months, reflecting LTA's administrative processing cycle. This is the foundational supply transmission mechanism.

**H2 — Bid Pressure Dominance:** The bid-to-quota ratio explains short-run premium movements better than quota volume alone. The intuition is that demand intensity relative to available supply — not supply in isolation — drives price.

**H3 — Spurious Levels Correlation:** Apparent correlations between supply variables and premiums in levels are trend-driven. In first differences, these relationships weaken substantially. This matters because policy commentary frequently cites level correlations as evidence of structural relationships.

**H4 — Regime Dependence:** The supply-premium relationship differs structurally across the pre- and post-population-stabilisation periods. If the stock constraint binds differently across regimes, the same quota change should have a different price impact depending on the period.

A fifth question — where exactly the structural break falls, and whether it is statistically identifiable — is addressed through formal break date testing before the main OLS estimation, since the regime dummy is a key regressor.

---

## 3. Data Pipeline

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.sandwich_covariance import cov_hac

from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0f0f",
    "axes.facecolor":   "#1a1a1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#ccc",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#eee",
    "grid.color":       "#2a2a2a",
    "grid.linestyle":   "--",
    "axes.grid":        True,
    "legend.facecolor": "#1a1a1a",
    "legend.edgecolor": "#444",
    "figure.dpi":       120,
})

ACCENT   = "#00bfff"
ACCENT2  = "#ff6b6b"
ACCENT3  = "#7bed9f"
ACCENT4  = "#ffa502"

print("Imports OK")

Imports OK


### 3.1 Data Sources

| Dataset | SingStat Table ID | Coverage | Key Fields |
|---|---|---|---|
| COE Bidding Results | `M651121` | Feb 2002 – present | quota, bids_received, successful_bids, quota_premium |
| Vehicle Deregistrations | `M650291` | May 1990 – present | monthly deregistrations by category |
| New Registrations | `M650281` | May 1990 – present | monthly new registrations by category |
| Vehicle Population | `M650341` | May 1990 – present | total vehicles on road by category |

All data is fetched from the SingStat TableBuilder API (`tablebuilder.singstat.gov.sg`). No rate limiting issues; no API key required.

In [ ]:
SINGSTAT_URL = "https://tablebuilder.singstat.gov.sg/api/table/tabledata"
HEADERS      = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}

def fetch_singstat(table_id: str, search: str = None, limit: int = 20000) -> dict:
    params = {"offset": 0, "limit": limit}
    if search:
        params["search"] = search
    resp = requests.get(f"{SINGSTAT_URL}/{table_id}", headers=HEADERS, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def singstat_to_long(data: dict) -> pd.DataFrame:
    records = []
    for row in data["Data"]["row"]:
        for col in row["columns"]:
            records.append({"month": col["key"], "metric": row["rowText"], "value": col["value"]})
    df = pd.DataFrame(records)
    df["month"] = pd.to_datetime(df["month"], format="%Y %b", errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df

def extract_cat_a_series(table_id: str, row_text: str, value_name: str) -> pd.DataFrame:
    data  = fetch_singstat(table_id)
    rows  = data["Data"]["row"]
    match = next((r for r in rows if r["rowText"].strip() == row_text), None)
    if match is None:
        raise ValueError(f"Row '{row_text}' not found in {table_id}")
    records = [{"month": c["key"], value_name: c["value"]} for c in match["columns"]]
    df = pd.DataFrame(records)
    df["month"]      = pd.to_datetime(df["month"], format="%Y %b", errors="coerce")
    df[value_name]   = pd.to_numeric(df[value_name], errors="coerce")
    return df.sort_values("month").reset_index(drop=True)


def fetch_coe() -> pd.DataFrame:
    data = fetch_singstat("M651121", search="1600cc")
    long = singstat_to_long(data)

    _FIELD_MAP = {
        "quota premium":   "quota_premium",
        "successful bids": "successful_bids",
        "bids received":   "bids_received",
        "quota,":          "quota",
        ", quota,":        "quota",
    }
    def parse_metric(s):
        s_low   = s.lower()
        field   = next((v for k, v in _FIELD_MAP.items() if k in s_low), None)
        # simpler: detect by keyword order
        if "quota premium" in s_low:    field = "quota_premium"
        elif "successful bids" in s_low: field = "successful_bids"
        elif "bids received" in s_low:   field = "bids_received"
        elif "quota" in s_low:           field = "quota"
        bidding = 1 if "1st" in s_low else (2 if "2nd" in s_low else None)
        return field, bidding

    rows = []
    for _, row in long.iterrows():
        field, bidding = parse_metric(row["metric"])
        if None in (field, bidding):
            continue
        rows.append({"month": row["month"], "bidding_no": bidding,
                     "field": field, "value": row["value"]})

    pivot = (pd.DataFrame(rows)
               .pivot_table(index=["month", "bidding_no"], columns="field",
                            values="value", aggfunc="mean")
               .reset_index())
    pivot.columns.name = None
    pivot["coe_category"] = "A"
    return pivot.sort_values(["month", "bidding_no"]).reset_index(drop=True)


coe_raw = fetch_coe()
print(f"COE data: {len(coe_raw):,} rows | {coe_raw['month'].min().date()} -> {coe_raw['month'].max().date()}")
coe_raw.tail(3)

In [ ]:
dereg_df  = extract_cat_a_series("M650291", "Category A: Cars", "dereg_a")
newreg_df = extract_cat_a_series("M650281", "Category A: Cars", "new_a")
pop_df    = extract_cat_a_series("M650341", "Category A: Cars", "pop_a")

print(f"Deregistrations  : {len(dereg_df):,} rows | {dereg_df['month'].min().date()} -> {dereg_df['month'].max().date()}")
print(f"New registrations: {len(newreg_df):,} rows | {newreg_df['month'].min().date()} -> {newreg_df['month'].max().date()}")
print(f"Vehicle population: {len(pop_df):,} rows | {pop_df['month'].min().date()} -> {pop_df['month'].max().date()}")

In [ ]:
coe_a = (
    coe_raw
    .groupby("month", as_index=False)
    .agg(
        quota           = ("quota",           "sum"),
        bids_received   = ("bids_received",   "sum"),
        successful_bids = ("successful_bids", "sum"),
        quota_premium   = ("quota_premium",   "mean"),
    )
)

df = (
    coe_a
    .merge(dereg_df[["month", "dereg_a"]], on="month", how="left")
    .merge(newreg_df[["month", "new_a"]],  on="month", how="left")
    .merge(pop_df[["month", "pop_a"]],     on="month", how="left")
    .sort_values("month")
    .reset_index(drop=True)
)

for col in ["quota", "bids_received", "successful_bids", "quota_premium"]:
    df[col] = df[col].replace(0, np.nan)

df = df.set_index("month")

print(f"Master dataset: {len(df):,} months | {df.index.min().date()} → {df.index.max().date()}")
print(f"Missing values:\n{df.isnull().sum()}")
df.tail(4)

### 3.2 Data Quality

Zero values in auction fields are replaced with `NaN` — a zero quota or zero bids indicates a missing or cancelled auction round. Trailing `NaN`s in the vehicle series reflect LTA's 1–2 month reporting lag relative to the COE series; these are expected and handled by dropping at point of use.

---

## 4. Feature Engineering

In [ ]:
df["bid_to_quota"]    = df["bids_received"] / df["quota"]
df["excess_demand"]   = df["bids_received"] - df["quota"]
df["net_supply_flow"] = df["new_a"] - df["dereg_a"]

for lag in [3, 6, 9, 12]:
    df[f"dereg_lag{lag}"] = df["dereg_a"].shift(lag)
    df[f"quota_lag{lag}"] = df["quota"].shift(lag)

df["log_premium"]     = np.log(df["quota_premium"])
df["premium_return"]  = df["log_premium"].diff()
df["d_quota_premium"] = df["quota_premium"].diff()

df["premium_vol_12m"] = df["premium_return"].rolling(12).std()
df["premium_zscore"]  = (
    (df["quota_premium"] - df["quota_premium"].rolling(24).mean())
    / df["quota_premium"].rolling(24).std()
)
df["btq_ma12"] = df["bid_to_quota"].rolling(12).mean()

df["regime"] = (df.index >= "2016-01-01").astype(int)  # 0 = growth phase, 1 = stable

df["d_quota"]        = df["quota"].diff()
df["d_bids"]         = df["bids_received"].diff()
df["d_dereg"]        = df["dereg_a"].diff()
df["d_new"]          = df["new_a"].diff()
df["d_bid_to_quota"] = df["bid_to_quota"].diff()

print("Features added. Shape:", df.shape)
df[["quota_premium", "bid_to_quota", "excess_demand", "net_supply_flow",
    "premium_return", "premium_vol_12m", "premium_zscore"]].describe().round(2)

---

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle("Figure 1 — COE Category A: Key Series Over Time", fontsize=13, y=1.01)

plot_df = df.dropna(subset=["quota_premium", "pop_a", "bid_to_quota", "premium_vol_12m"])

ax = axes[0]
ax.plot(plot_df.index, plot_df["quota_premium"] / 1000, color=ACCENT, lw=1.5)
ax.axvspan(pd.Timestamp("2016-01-01"), plot_df.index.max(),
           alpha=0.08, color=ACCENT3, label="Stable population regime (post-2016)")
ax.set_ylabel("COE Premium (S$'000)")
ax.set_title("Quota Premium", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(plot_df.index, plot_df["pop_a"] / 1000, color=ACCENT3, lw=1.5)
ax.axvspan(pd.Timestamp("2016-01-01"), plot_df.index.max(), alpha=0.08, color=ACCENT3)
ax.set_ylabel("Vehicles ('000)")
ax.set_title("Cat A Vehicle Population", fontsize=10)

ax = axes[2]
ax.plot(plot_df.index, plot_df["bid_to_quota"], color=ACCENT4, lw=1.2, alpha=0.7, label="Monthly")
ax.plot(plot_df.index, plot_df["btq_ma12"],     color=ACCENT4, lw=2.0, label="12m MA")
ax.axhline(1.0, color="#666", lw=0.8, ls="--")
ax.set_ylabel("Bids / Quota")
ax.set_title("Bid-to-Quota Ratio", fontsize=10)
ax.legend(fontsize=8)

ax = axes[3]
ax.fill_between(plot_df.index, plot_df["premium_vol_12m"], color=ACCENT2, alpha=0.6)
ax.set_ylabel("Std Dev (log returns)")
ax.set_title("12-Month Rolling Premium Volatility", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Figure 2 — Supply Transmission Chain: Deregistrations → Quota → Bids", fontsize=13)

supply_df = df.dropna(subset=["dereg_a", "quota", "bids_received", "quota_premium"])

ax = axes[0]
ax.plot(supply_df.index, supply_df["dereg_a"], color=ACCENT2, lw=1.4, label="Deregistrations")
ax.plot(supply_df.index, supply_df["quota"],   color=ACCENT,  lw=1.4, label="Quota allocated")
ax.set_ylabel("Vehicles")
ax.set_title("Deregistrations vs Quota", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(supply_df.index, supply_df["bids_received"], color=ACCENT4, lw=1.4, label="Bids received")
ax.plot(supply_df.index, supply_df["quota"],         color=ACCENT,  lw=1.4, label="Quota", alpha=0.7)
ax.set_ylabel("Vehicles")
ax.set_title("Bids Received vs Quota", fontsize=10)
ax.legend(fontsize=8)

ax = axes[2]
ax2 = ax.twinx()
ax.bar(supply_df.index, supply_df["excess_demand"], color=ACCENT4, alpha=0.4,
       width=20, label="Excess demand (bids − quota)")
ax2.plot(supply_df.index, supply_df["quota_premium"] / 1000, color=ACCENT, lw=1.5,
         label="Premium (S$'000)")
ax.set_ylabel("Excess Demand (vehicles)", color=ACCENT4)
ax2.set_ylabel("Premium (S$'000)", color=ACCENT)
ax.set_title("Excess Demand vs Premium", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Figure 3 — Regime Comparison: Pre-2016 vs Post-2016", fontsize=13)

regime_df = df.dropna(subset=["quota_premium", "bid_to_quota", "premium_vol_12m"])
pre  = regime_df[regime_df["regime"] == 0]
post = regime_df[regime_df["regime"] == 1]

for ax, col, label, scale in zip(
    axes,
    ["quota_premium", "bid_to_quota", "premium_vol_12m"],
    ["COE Premium (S$)", "Bid-to-Quota Ratio", "12m Rolling Volatility"],
    [1/1000, 1, 1]
):
    ax.hist(pre[col]  * scale, bins=25, color=ACCENT,  alpha=0.6, label="Pre-2016",  density=True)
    ax.hist(post[col] * scale, bins=25, color=ACCENT2, alpha=0.6, label="Post-2016", density=True)
    ax.set_xlabel(label)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    pre_med  = (pre[col]  * scale).median()
    post_med = (post[col] * scale).median()
    ax.axvline(pre_med,  color=ACCENT,  lw=1.5, ls="--")
    ax.axvline(post_med, color=ACCENT2, lw=1.5, ls="--")
    ax.set_title(f"Median: {pre_med:.1f} → {post_med:.1f}", fontsize=9)

plt.tight_layout()
plt.show()

print("\nRegime summary statistics:")
print(regime_df.groupby("regime")[["quota_premium", "bid_to_quota", "premium_vol_12m", "excess_demand"]]
      .agg(["mean", "std", "median"]).round(2))

In [ ]:
def lagged_spearman(x: pd.Series, y: pd.Series, max_lag: int = 15) -> pd.DataFrame:
    results = []
    for lag in range(0, max_lag + 1):
        y_lagged = y.shift(lag)
        mask = x.notna() & y_lagged.notna()
        if mask.sum() < 20:
            continue
        r, p = spearmanr(x[mask], y_lagged[mask])
        results.append({"lag": lag, "rho": r, "pvalue": p})
    return pd.DataFrame(results)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 4 — Lagged Spearman Correlations", fontsize=13)

lc1 = lagged_spearman(df["quota"], df["dereg_a"], max_lag=15)
ax  = axes[0]
ax.bar(lc1["lag"], lc1["rho"],
       color=[ACCENT if p < 0.05 else "#555" for p in lc1["pvalue"]])
ax.axhline(0, color="#666", lw=0.8)
ax.set_xlabel("Lag (months) — dereg leads quota")
ax.set_ylabel("Spearman ρ")
ax.set_title("Deregistrations → Quota (H1: expect peak at 6–9 months)", fontsize=10)
ax.set_xticks(lc1["lag"])

lc2 = lagged_spearman(df["premium_return"], df["bid_to_quota"], max_lag=15)
ax  = axes[1]
ax.bar(lc2["lag"], lc2["rho"],
       color=[ACCENT2 if p < 0.05 else "#555" for p in lc2["pvalue"]])
ax.axhline(0, color="#666", lw=0.8)
ax.set_xlabel("Lag (months) — bid_to_quota leads premium_return")
ax.set_ylabel("Spearman ρ")
ax.set_title("Bid-to-Quota → Premium Return (H2: expect significant at lag 0–2)", fontsize=10)
ax.set_xticks(lc2["lag"])

for ax, lc in zip(axes, [lc1, lc2]):
    peak = lc.loc[lc["rho"].abs().idxmax()]
    ax.annotate(f"Peak: lag {int(peak['lag'])}\nρ={peak['rho']:.2f}",
                xy=(peak["lag"], peak["rho"]),
                xytext=(peak["lag"] + 1.5, peak["rho"] * 0.85),
                fontsize=8, color="#eee",
                arrowprops=dict(arrowstyle="->", color="#aaa", lw=0.8))

plt.tight_layout()
plt.show()

---

## 6. Statistical and Econometric Analysis

The analysis proceeds in a deliberate sequence. We first establish the time-series properties of each variable (stationarity), then test the supply transmission chain (Granger causality), then identify the structural break date before imposing it as a regime dummy, and finally estimate the OLS model with HAC standard errors. COVID-19 is treated as a discrete shock with dummy variables rather than a separate sub-sample, preserving degrees of freedom while controlling for the distortion it introduces into the premium series.

In [ ]:
def adf_summary(series_dict: dict) -> pd.DataFrame:
    rows = []
    for name, s in series_dict.items():
        s_clean = s.dropna()
        if len(s_clean) < 20:
            continue
        stat, pval, _, nobs, crit, _ = adfuller(s_clean, autolag="AIC")
        rows.append({
            "Series":          name,
            "ADF Stat":        round(stat, 3),
            "p-value":         round(pval, 4),
            "N":               nobs,
            "1% CV":           round(crit["1%"], 3),
            "Stationary (5%)": "Yes" if pval < 0.05 else "No"
        })
    return pd.DataFrame(rows).set_index("Series")


levels_series = {
    "quota_premium": df["quota_premium"],
    "log_premium":   df["log_premium"],
    "quota":         df["quota"],
    "bids_received": df["bids_received"],
    "bid_to_quota":  df["bid_to_quota"],
    "dereg_a":       df["dereg_a"],
    "new_a":         df["new_a"],
    "pop_a":         df["pop_a"],
}

diff_series = {
    "d_quota_premium": df["d_quota_premium"],
    "premium_return":  df["premium_return"],
    "d_quota":         df["d_quota"],
    "d_bids":          df["d_bids"],
    "d_bid_to_quota":  df["d_bid_to_quota"],
    "d_dereg":         df["d_dereg"],
    "d_new":           df["d_new"],
}

print("=== ADF Tests: Levels ===")
print(adf_summary(levels_series).to_string())
print("\n=== ADF Tests: First Differences ===")
print(adf_summary(diff_series).to_string())

In [ ]:
def spearman_matrix(data: pd.DataFrame, cols: list) -> tuple:
    n       = len(cols)
    rho_mat = np.zeros((n, n))
    p_mat   = np.zeros((n, n))
    clean   = data[cols].dropna()
    for i, c1 in enumerate(cols):
        for j, c2 in enumerate(cols):
            r, p = spearmanr(clean[c1], clean[c2])
            rho_mat[i, j] = r
            p_mat[i, j]   = p
    return (pd.DataFrame(rho_mat, index=cols, columns=cols),
            pd.DataFrame(p_mat,   index=cols, columns=cols))


level_cols = ["quota_premium", "quota", "bids_received", "bid_to_quota",
              "dereg_a", "new_a", "pop_a"]
diff_cols  = ["premium_return", "d_quota", "d_bids", "d_bid_to_quota",
              "d_dereg", "d_new"]

rho_lev, _ = spearman_matrix(df, level_cols)
rho_dif, _ = spearman_matrix(df, diff_cols)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Figure 5 — Spearman Correlation: Levels vs First Differences (H3)", fontsize=13)

for ax, rho, title in zip(axes,
                           [rho_lev, rho_dif],
                           ["Levels (likely spurious — shared trends)",
                            "First Differences (structural signal)"]):
    mask = np.triu(np.ones_like(rho, dtype=bool), k=1)
    sns.heatmap(rho, ax=ax, annot=True, fmt=".2f", cmap="RdBu_r",
                vmin=-1, vmax=1, linewidths=0.4,
                annot_kws={"size": 8}, mask=mask,
                cbar_kws={"shrink": 0.8})
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", rotation=0,  labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def granger_table(df_g: pd.DataFrame, pairs: list, max_lag: int = 6) -> pd.DataFrame:
    rows = []
    for caused, causing in pairs:
        data = df_g[[caused, causing]].dropna()
        if len(data) < max_lag + 10:
            continue
        try:
            res = grangercausalitytests(data[[caused, causing]], maxlag=max_lag, verbose=False)
            best_lag, best_p = min(
                ((lag, res[lag][0]["ssr_ftest"][1]) for lag in range(1, max_lag + 1)),
                key=lambda x: x[1]
            )
            rows.append({
                "Caused":            caused,
                "Causing":           causing,
                "Best lag":          best_lag,
                "Min p-val":         round(best_p, 4),
                "Significant (5%)":  "Yes" if best_p < 0.05 else "No"
            })
        except Exception:
            pass
    return pd.DataFrame(rows)


granger_df = df[["premium_return", "d_quota_premium",
                  "d_quota", "d_bids", "d_bid_to_quota",
                  "d_dereg", "d_new"]].dropna()

pairs = [
    ("d_quota",        "d_dereg"),         # H1: dereg → quota
    ("d_bids",         "d_quota"),         # quota → bids
    ("premium_return", "d_bids"),          # bids → premium
    ("premium_return", "d_bid_to_quota"),  # H2: bid pressure → premium
    ("premium_return", "d_quota"),         # quota → premium (direct)
    ("premium_return", "d_dereg"),         # dereg → premium (direct)
    ("premium_return", "d_new"),           # new registrations → premium
    ("d_new",          "d_quota"),         # quota → new registrations
]

gc_results = granger_table(granger_df, pairs, max_lag=6)
print("=== Granger Causality Summary ===")
print(gc_results.to_string(index=False))

In [ ]:
reg_df = df[["premium_return", "d_bid_to_quota", "d_quota", "d_dereg", "regime"]].copy()

for lag in [1, 2, 3]:
    reg_df[f"d_btq_lag{lag}"] = reg_df["d_bid_to_quota"].shift(lag)

reg_df = reg_df.dropna()
print(f"OLS sample: {len(reg_df)} observations ({reg_df.index.min().date()} → {reg_df.index.max().date()})")

y = reg_df["premium_return"]
X = sm.add_constant(reg_df[["d_bid_to_quota", "d_btq_lag1", "d_btq_lag2",
                              "d_quota", "d_dereg", "regime"]])

ols_model = OLS(y, X).fit()
ols_hac   = ols_model.get_robustcov_results(cov_type="HAC", maxlags=6)

print("\n=== OLS with HAC Standard Errors (Newey-West, 6 lags) ===")
print(ols_hac.summary2(float_format="%.4f"))

In [ ]:
from scipy.stats import norm

resid = pd.Series(np.asarray(ols_hac.resid), index=reg_df.index, name="resid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Figure 6 — OLS Residual Diagnostics", fontsize=12)

axes[0].plot(resid.index, resid, color=ACCENT, lw=0.8, alpha=0.8)
axes[0].axhline(0, color="#666", lw=0.8, ls="--")
axes[0].set_title("Residuals over time")
axes[0].set_ylabel("Residual")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[1].hist(resid, bins=30, color=ACCENT, alpha=0.7, density=True)
xr = np.linspace(resid.min(), resid.max(), 200)
axes[1].plot(xr, norm.pdf(xr, resid.mean(), resid.std()), color=ACCENT2, lw=1.5)
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual")

acf_vals = acf(resid.dropna(), nlags=20, alpha=0.05)
acf_coef, acf_ci = acf_vals[0], acf_vals[1]
lags = np.arange(len(acf_coef))
axes[2].bar(lags, acf_coef, color=ACCENT, alpha=0.7)
axes[2].fill_between(lags, acf_ci[:, 0] - acf_coef, acf_ci[:, 1] - acf_coef,
                     alpha=0.2, color=ACCENT2)
axes[2].axhline(0, color="#666", lw=0.8)
axes[2].set_title("ACF of residuals")
axes[2].set_xlabel("Lag")

plt.tight_layout()
plt.show()

lb = acorr_ljungbox(resid.dropna(), lags=[6, 12], return_df=True)
print("\nLjung-Box test on residuals:")
print(lb.round(4))

bp_stat, bp_p, _, _ = het_breuschpagan(resid, X)
print(f"\nBreusch-Pagan test: stat={bp_stat:.3f}, p={bp_p:.4f}",
      "-> heteroskedasticity present" if bp_p < 0.05 else "-> no significant heteroskedasticity")

In [ ]:
roll_window = 36
roll_df = df[["premium_return", "d_bid_to_quota", "d_quota"]].dropna().copy()

coefs, dates, r2s = [], [], []
for i in range(roll_window, len(roll_df) + 1):
    window = roll_df.iloc[i - roll_window: i]
    y_w = window["premium_return"]
    X_w = sm.add_constant(window[["d_bid_to_quota", "d_quota"]])
    try:
        m = OLS(y_w, X_w).fit()
        coefs.append(m.params["d_bid_to_quota"])
        r2s.append(m.rsquared)
        dates.append(window.index[-1])
    except Exception:
        pass

roll_coef_df = pd.DataFrame({"date": dates, "btq_coef": coefs, "r2": r2s}).set_index("date")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle("Figure 7 — Rolling OLS (36-month window): Structural Stability (H4)", fontsize=12)

ax = axes[0]
ax.plot(roll_coef_df.index, roll_coef_df["btq_coef"], color=ACCENT, lw=1.5)
ax.axhline(0, color="#666", lw=0.8, ls="--")
ax.axvspan(pd.Timestamp("2016-01-01"), roll_coef_df.index.max(),
           alpha=0.08, color=ACCENT3, label="Post-2016 regime")
ax.set_ylabel("Coefficient on Δ(bid-to-quota)")
ax.set_title("Rolling coefficient: bid pressure → premium return", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.fill_between(roll_coef_df.index, roll_coef_df["r2"], color=ACCENT4, alpha=0.5)
ax.axvspan(pd.Timestamp("2016-01-01"), roll_coef_df.index.max(),
           alpha=0.08, color=ACCENT3)
ax.set_ylabel("R²")
ax.set_title("Rolling R²", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

### 6.7 COVID-19 Shock

COVID-19 was not a standard demand shock. It was a simultaneous supply and demand disruption: LTA suspended auction rounds, showrooms closed, and pent-up demand accumulated over 16 months. The post-COVID rebound drove Cat A premiums from ~S$30k in mid-2020 to over S$100k by 2023 — the sharpest sustained rally in the 24-year sample.

Treating this period as business-as-usual would contaminate the OLS estimates. Two dummy variables are added: one for the disruption period (Feb 2020 – Jun 2021) and one for the rebound phase (Jul 2021 – Jun 2023). This isolates the COVID effect rather than absorbing it into the regime or quota coefficients, and allows us to test whether the structural supply-premium relationship changed after the shock resolved.

In [ ]:
# ── COVID shock: define periods and measure impact ───────────────────────────

covid_start  = pd.Timestamp("2020-02-01")
covid_trough = pd.Timestamp("2020-07-01")
covid_end    = pd.Timestamp("2021-06-01")
rebound_peak = pd.Timestamp("2023-06-01")

pre_covid  = df[df.index <  covid_start]
covid_dip  = df[(df.index >= covid_start) & (df.index <= covid_end)]
post_covid = df[df.index >  covid_end]

print("=== COVID-19 Shock: Premium Statistics by Period ===\n")
for label, subset in [("Pre-COVID  (Feb 2002 – Jan 2020)", pre_covid),
                       ("COVID dip  (Feb 2020 – Jun 2021)", covid_dip),
                       ("Post-COVID (Jul 2021 – Feb 2026)", post_covid)]:
    p = subset["quota_premium"].dropna()
    print(f"  {label}")
    print(f"    Mean: S${p.mean():,.0f}  |  Median: S${p.median():,.0f}  |  "
          f"Min: S${p.min():,.0f}  |  Max: S${p.max():,.0f}\n")

# ── Figure 11: COVID shock annotation ────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle("Figure 11 — COVID-19 Shock: Premium and Volatility", fontsize=13)

prem = df["quota_premium"].dropna()
vol  = df["premium_vol_12m"].dropna()

ax = axes[0]
ax.plot(prem.index, prem / 1000, color=ACCENT, lw=1.5)
ax.axvspan(covid_start, covid_end,    alpha=0.15, color=ACCENT2, label="COVID disruption period")
ax.axvspan(covid_end,   rebound_peak, alpha=0.08, color=ACCENT4, label="Pent-up demand rebound")
ax.axhline(pre_covid["quota_premium"].mean()  / 1000, color="#555", lw=0.8, ls=":",
           label=f"Pre-COVID mean (S${pre_covid['quota_premium'].mean()/1000:.0f}k)")
ax.axhline(post_covid["quota_premium"].mean() / 1000, color=ACCENT3, lw=0.8, ls=":",
           label=f"Post-COVID mean (S${post_covid['quota_premium'].mean()/1000:.0f}k)")
ax.set_ylabel("COE Premium (S$'000)")
ax.set_title("Quota Premium with COVID Shock Annotation", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.fill_between(vol.index, vol, color=ACCENT2, alpha=0.5)
ax.axvspan(covid_start, covid_end,    alpha=0.15, color=ACCENT2)
ax.axvspan(covid_end,   rebound_peak, alpha=0.08, color=ACCENT4)
ax.set_ylabel("Std Dev (log returns)")
ax.set_title("12-Month Rolling Volatility — COVID spike visible", fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

# ── COVID dummy in OLS ────────────────────────────────────────────────────────
reg_covid = df[["premium_return", "d_quota", "d_bid_to_quota", "regime"]].copy()
reg_covid["covid"]               = ((df.index >= covid_start) & (df.index <= covid_end)).astype(int)
reg_covid["post_covid_rebound"]  = ((df.index > covid_end) & (df.index <= rebound_peak)).astype(int)
reg_covid = reg_covid.dropna()

y_c  = reg_covid["premium_return"]
X_c  = sm.add_constant(reg_covid[["d_quota", "d_bid_to_quota", "regime",
                                    "covid", "post_covid_rebound"]])
ols_covid = OLS(y_c, X_c).fit().get_robustcov_results(cov_type="HAC", maxlags=6)

print("=== OLS with COVID Dummies (HAC SE) ===")
print(f"{'Variable':<22} {'Coef':>10} {'p-value':>10}")
print("-" * 44)
params  = ols_covid.params
pvalues = ols_covid.pvalues
for i, var in enumerate(ols_covid.model.exog_names):
    print(f"  {var:<20} {params[i]:>10.5f} {pvalues[i]:>10.4f}")

baseline_r2 = OLS(y_c, sm.add_constant(
    reg_covid[["d_quota", "d_bid_to_quota", "regime"]]
)).fit().rsquared
print(f"\n  R² = {ols_covid.rsquared:.4f}  (vs {baseline_r2:.4f} without COVID dummies)")

### 6.8 OLS Specification Note

The main OLS model regresses monthly log premium returns on: Δquota, Δbid-to-quota, the regime dummy (set to the grid-search break date), a COVID disruption dummy, and a post-COVID rebound dummy. HAC standard errors (Newey-West, 6 lags) correct for the residual autocorrelation confirmed by the Ljung-Box test. The model is estimated in first differences throughout — levels are non-stationary (ADF p=0.927 for quota_premium) and level regressions would produce spurious results.

The regime dummy captures the mean shift in the supply-premium relationship across the two periods. It does not impose that the *slope* coefficients change — that is tested separately via the Chow test and rolling OLS. If the Chow test confirms slope instability, the correct specification is an interacted model; if only the intercept shifts, the additive dummy suffices.

In [ ]:
from statsmodels.stats.diagnostic import breaks_cusumolsresid
import itertools

# ── Chow test at 2016-01-01 ───────────────────────────────────────────────────
# Tests whether the OLS coefficients are stable across the 2016 break

chow_df = df[["premium_return", "d_quota", "d_bid_to_quota"]].dropna().copy()
chow_df["post2016"] = (chow_df.index >= "2016-01-01").astype(int)

# Interact all regressors with the regime dummy
for col in ["d_quota", "d_bid_to_quota"]:
    chow_df[f"{col}_x_post"] = chow_df[col] * chow_df["post2016"]

y_chow = chow_df["premium_return"]
X_restricted   = sm.add_constant(chow_df[["d_quota", "d_bid_to_quota"]])
X_unrestricted = sm.add_constant(chow_df[["d_quota", "d_bid_to_quota", "post2016",
                                           "d_quota_x_post", "d_bid_to_quota_x_post"]])

ols_r  = OLS(y_chow, X_restricted).fit()
ols_ur = OLS(y_chow, X_unrestricted).fit()

# Chow F-statistic: ((RSS_r - RSS_ur) / k) / (RSS_ur / (n - 2k))
n  = len(y_chow)
k  = X_restricted.shape[1]   # number of params in restricted model
rss_r  = ols_r.ssr
rss_ur = ols_ur.ssr
chow_f = ((rss_r - rss_ur) / k) / (rss_ur / (n - 2 * k))
from scipy.stats import f as f_dist
chow_p = 1 - f_dist.cdf(chow_f, k, n - 2 * k)

print("=== Chow Test: Structural Break at 2016-01-01 ===")
print(f"  F-statistic : {chow_f:.4f}")
print(f"  p-value     : {chow_p:.4f}")
print(f"  Verdict     : {'Break confirmed (p<0.05)' if chow_p < 0.05 else 'Break NOT confirmed (p>=0.05)'}")

# ── CUSUM test ────────────────────────────────────────────────────────────────
cusum_result = breaks_cusumolsresid(ols_r.resid)
print(f"\n=== CUSUM Test (OLS residuals) ===")
print(f"  Statistic: {cusum_result[0]:.4f}  |  p-value: {cusum_result[1]:.4f}")
print(f"  Verdict  : {'Structural instability detected' if cusum_result[1] < 0.05 else 'No significant instability'}")

# ── Grid search: which break date minimises RSS? ─────────────────────────────
# Test every candidate break date between 2008 and 2022 (min 36 obs each side)
candidates = pd.date_range("2008-01-01", "2022-01-01", freq="QS")
results = []
for bp in candidates:
    d = chow_df.copy()
    d["post"] = (d.index >= bp).astype(int)
    for col in ["d_quota", "d_bid_to_quota"]:
        d[f"{col}_x_post"] = d[col] * d["post"]
    X_bp = sm.add_constant(d[["d_quota", "d_bid_to_quota", "post",
                                "d_quota_x_post", "d_bid_to_quota_x_post"]])
    if X_bp.shape[0] < 50:
        continue
    try:
        m = OLS(y_chow, X_bp).fit()
        results.append({"break_date": bp, "rss": m.ssr, "aic": m.aic, "bic": m.bic})
    except Exception:
        pass

bp_df = pd.DataFrame(results).set_index("break_date")
best_rss = bp_df["rss"].idxmin()
best_aic = bp_df["aic"].idxmin()
best_bic = bp_df["bic"].idxmin()

print(f"\n=== Grid Search: Optimal Break Date ===")
print(f"  Minimum RSS : {best_rss.date()}  (RSS={bp_df.loc[best_rss,'rss']:.6f})")
print(f"  Minimum AIC : {best_aic.date()}  (AIC={bp_df.loc[best_aic,'aic']:.4f})")
print(f"  Minimum BIC : {best_bic.date()}  (BIC={bp_df.loc[best_bic,'bic']:.4f})")
print(f"\n  Assumed break (2016-01-01) RSS : {bp_df.loc[pd.Timestamp('2016-01-01'),'rss']:.6f}"
      if pd.Timestamp("2016-01-01") in bp_df.index else "")

# ── Figure 12: RSS profile across candidate break dates ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 12 — Break Date Search: RSS and BIC Profile", fontsize=12)

ax = axes[0]
ax.plot(bp_df.index, bp_df["rss"], color=ACCENT, lw=1.5)
ax.axvline(best_rss, color=ACCENT2, lw=1.5, ls="--", label=f"Min RSS: {best_rss.date()}")
ax.axvline(pd.Timestamp("2016-01-01"), color=ACCENT3, lw=1.2, ls=":", label="Assumed: 2016-01-01")
ax.set_ylabel("Residual Sum of Squares")
ax.set_title("RSS by Break Date", fontsize=10)
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax = axes[1]
ax.plot(bp_df.index, bp_df["bic"], color=ACCENT4, lw=1.5)
ax.axvline(best_bic, color=ACCENT2, lw=1.5, ls="--", label=f"Min BIC: {best_bic.date()}")
ax.axvline(pd.Timestamp("2016-01-01"), color=ACCENT3, lw=1.2, ls=":", label="Assumed: 2016-01-01")
ax.set_ylabel("BIC")
ax.set_title("BIC by Break Date", fontsize=10)
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

### 6.6 Break Date Findings

Three tests were run to determine whether the 2016 population stabilisation represents a statistically identifiable structural break in the supply-premium relationship.

**Chow test at 2016-01-01** tests whether the OLS coefficients on quota changes and bid pressure are stable across the assumed break. A significant F-statistic confirms the coefficients differ across sub-periods, validating the regime split as more than an economic narrative. An insignificant result would mean the 2016 dummy adds no explanatory power and should be dropped.

**CUSUM test** examines whether the recursive residuals from the restricted OLS drift systematically over time. Rejection of the null indicates parameter instability somewhere in the sample — consistent with at least one structural break, though it does not identify the date.

**Grid search (RSS and BIC minimisation)** scans every candidate break date from 2008 to 2022 in quarterly steps. The RSS-minimising date is where splitting the sample most reduces unexplained variance. The BIC-minimising date applies a parsimony penalty, giving a more conservative estimate. A sharp trough in the profile indicates a well-identified discrete break; a flat profile suggests gradual structural drift rather than a single shift.

The regime dummy used in the OLS (Section 6.8) is set to the data-preferred break date from the grid search. Where the grid search and the economic narrative (2016 population stabilisation) agree, the identification is robust. Where they diverge, the grid search date takes precedence — the data should discipline the prior.

---

## 7. Forecasting

Short-run predictability of premium returns is tested first via walk-forward validation. Three models are evaluated on the final 20% of the sample (Jun 2021 – Feb 2026):

1. **Naive** — random walk; zero return forecast each period. The appropriate benchmark for any financial time series.
2. **AR(1)** — autoregressive model on premium returns. Tests whether past returns predict future returns.
3. **ARIMAX(1,0,0)** — AR(1) augmented with Δbid-to-quota as an exogenous regressor. Tests whether the contemporaneous demand signal adds predictive power beyond the autoregressive component.

If the ARIMAX cannot beat the naive benchmark on RMSE, the implication is clear: short-run premium movements are not forecastable from public administrative data. The 5-year scenario framework in Section 10 is therefore built on structural and institutional drivers — the cohort deregistration wave, regime-level demand assumptions, and macro tail risks — rather than time-series extrapolation.

In [ ]:
fc_df = df[["premium_return", "d_bid_to_quota"]].dropna().copy()

n       = len(fc_df)
n_train = int(n * 0.80)
n_test  = n - n_train

print(f"Total: {n} | Train: {n_train} | Test: {n_test}")
print(f"Test period: {fc_df.index[n_train].date()} → {fc_df.index[-1].date()}")

y_all    = fc_df["premium_return"].values
exog_all = fc_df["d_bid_to_quota"].values

naive_preds, ar1_preds, arimax_preds, actuals = [], [], [], []

for t in range(n_train, n):
    y_train    = y_all[:t]
    exog_train = exog_all[:t]
    exog_next  = exog_all[t: t + 1]

    naive_preds.append(0.0)

    try:
        ar1 = SARIMAX(y_train, order=(1, 0, 0), trend="c").fit(disp=False)
        ar1_preds.append(float(ar1.forecast(1)))
    except Exception:
        ar1_preds.append(0.0)

    try:
        arimax = SARIMAX(y_train, exog=exog_train, order=(1, 0, 0), trend="c").fit(disp=False)
        arimax_preds.append(float(arimax.forecast(1, exog=exog_next)))
    except Exception:
        arimax_preds.append(ar1_preds[-1])

    actuals.append(y_all[t])

actuals      = np.array(actuals)
naive_preds  = np.array(naive_preds)
ar1_preds    = np.array(ar1_preds)
arimax_preds = np.array(arimax_preds)

def eval_metrics(y_true, y_pred, name):
    return {
        "Model":    name,
        "RMSE":     round(np.sqrt(mean_squared_error(y_true, y_pred)), 5),
        "MAE":      round(mean_absolute_error(y_true, y_pred), 5),
        "MAPE (%)": round(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100, 2),
    }

metrics = pd.DataFrame([
    eval_metrics(actuals, naive_preds,  "Naive (RW)"),
    eval_metrics(actuals, ar1_preds,    "AR(1)"),
    eval_metrics(actuals, arimax_preds, "ARIMAX(1,0,0) + Δbtq"),
])
print("\n=== Out-of-Sample Forecast Evaluation ===")
print(metrics.to_string(index=False))

In [ ]:
test_dates = fc_df.index[n_train:]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle("Figure 8 — Walk-Forward Forecast: Premium Returns (Out-of-Sample)", fontsize=12)

ax = axes[0]
ax.plot(test_dates, actuals,      color="#aaa",  lw=1.2, label="Actual",        alpha=0.9)
ax.plot(test_dates, ar1_preds,    color=ACCENT,  lw=1.2, label="AR(1)",         alpha=0.85)
ax.plot(test_dates, arimax_preds, color=ACCENT2, lw=1.2, label="ARIMAX + Δbtq", alpha=0.85)
ax.axhline(0, color="#555", lw=0.7, ls="--")
ax.set_ylabel("Log return")
ax.set_title("Predicted vs Actual Premium Returns", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(test_dates, np.cumsum((actuals - naive_preds)  ** 2), color="#888",  lw=1.2, label="Naive (RW)")
ax.plot(test_dates, np.cumsum((actuals - ar1_preds)    ** 2), color=ACCENT,  lw=1.2, label="AR(1)")
ax.plot(test_dates, np.cumsum((actuals - arimax_preds) ** 2), color=ACCENT2, lw=1.2, label="ARIMAX + Δbtq")
ax.set_ylabel("Cumulative Squared Error")
ax.set_title("Cumulative Forecast Error", fontsize=10)
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

---

## 8. Interpretation

### Supply Is the Mechanism, Not the Signal

The Granger results confirm the supply transmission chain: deregistrations → quota (lag 6, p=0.003) → bids (p<0.001) → premium returns (p=0.021). Each link in the chain is statistically identifiable. The OLS coefficient on Δquota (−0.0002, p<0.001, HAC SE) implies a 1,000-unit increase in monthly quota is associated with a −2% log return in premium. This is the dominant short-run predictor in the model.

This is not surprising once you understand the mechanism. Quota is the only supply-side variable that moves independently of contemporaneous demand — it is set administratively, based on past deregistrations, before the auction opens. It is therefore the closest thing to an exogenous supply shock available in the public data. That it predicts premium returns is consistent with basic supply-demand theory applied to an administratively constrained market.

### Why the Bid-to-Quota Ratio Fails

H2 is rejected, but the reason matters. The bid-to-quota ratio is stationary and mean-reverting (ADF p<0.001), which mechanically limits its Granger predictive power for premium *returns*. But the deeper issue is economic: the ratio is an equilibrium outcome, not a forcing variable. It reflects the market-clearing condition at each auction — by definition, it adjusts simultaneously with price. Regressing future premium returns on a contemporaneous equilibrium ratio is asking whether the market's current clearing condition predicts where it will clear next. It does not, because the ratio already incorporates all available information at the time of clearing.

The correct interpretation is that quota changes — which are predetermined relative to the auction — carry the predictive signal, while the bid-to-quota ratio is a contemporaneous description of market tightness. Both matter economically; only one is useful for prediction.

### Levels Correlations Are a Red Herring

The strong negative level-correlation between quota and premiums during the growth phase reflects a shared trend, not a structural relationship. Both series were trending — quota upward as the vehicle population grew, premiums upward as demand outpaced supply — and their correlation is largely spurious. ADF confirms the premium series has a unit root in levels (p=0.927). In first differences, the relationship weakens substantially and the OLS R² is just 8%. This is the correct number to cite: public supply and auction data explain roughly one-twelfth of monthly premium return variance.

### The Regime Shift Is Real, But Its Cause Is Compositional

Post-break, mean premium rose 77% while monthly volatility halved. The rolling OLS confirms the bid pressure coefficient shifted after the break. The economic interpretation is compositional: once the vehicle population stabilised, the marginal buyer changed. The growth-phase market included many first-time buyers with high price sensitivity. The stable-population market is dominated by replacement buyers — households renewing an existing car — whose demand is less elastic because the alternative is losing car access entirely. Higher inelastic demand at the margin means the same quota reduction produces a larger price response, and the same quota increase produces a smaller relief. Mean premium rises; volatility falls because the demand curve is flatter in the relevant range.

### A Note on Causality

Granger causality is predictive ordering, not causal identification. The OLS estimates are reduced-form associations, not structural parameters. Both quota changes and premium returns may respond to common unobserved drivers — consumer confidence, credit conditions, macroeconomic sentiment. A credible causal claim requires an instrument: exogenous variation in quota uncorrelated with contemporaneous demand. LTA's discretionary adjustments are a candidate but are not separately identified in the public data. The findings should be read as consistent with the supply-transmission hypothesis, not as proof of it.

---

## 9. Limitations and Extensions

### What This Analysis Cannot Do

**No causal identification.** The OLS estimates are reduced-form associations. Identifying the causal impact of quota on premiums requires an instrument — exogenous variation in quota uncorrelated with contemporaneous demand. LTA's discretionary adjustments are a candidate but are not separately identified in the public data. The Granger results establish predictive ordering; they do not establish causation.

**Demand proxies are noisy.** Bids received reflects the number of bidders above the reserve price, not the full willingness-to-pay distribution. Shifts in bidding strategy — conservative bidding during uncertain periods, aggressive overbidding when buyers fear missing out — can move bids received independently of underlying demand. The bid-to-quota ratio inherits this noise.

**Omitted demand-side variables.** The model excludes car loan interest rates, household income, HDB resale prices (a wealth proxy), MAS loan-to-value rules, and cross-category substitution. A Cat A buyer priced out at S$108k does not disappear — they wait, switch to Cat B, or take the MRT. None of that substitution is captured here. Augmenting with these variables would likely improve R² substantially and sharpen the regime interpretation.

**Single-equation framework.** Estimating Cat A in isolation ignores cross-category dynamics. Cat A, B, and E premiums are jointly determined — a supply shock in one category shifts demand into others. A VAR across categories would capture these spillovers.

**Low R².** The OLS explains 8% of monthly premium return variance. This is not a model failure — it reflects the genuine limits of what predetermined supply variables can identify in a market where most of the variance is driven by unobservable demand shocks. It is, however, a strong argument against using this model for short-run trading or timing decisions.

### Where to Go Next

1. Augment with car loan rates, household income, and HDB resale prices as demand-side controls.
2. Estimate a VAR across Cat A, B, and E to capture cross-category substitution.
3. Apply GARCH(1,1) to premium returns — the rolling volatility analysis confirms time-varying conditional variance.
4. Use Google Trends search volume for "COE Singapore" as a higher-frequency, forward-looking demand proxy.
5. Implement Bai-Perron multiple breakpoint testing to identify all regime changes endogenously, rather than testing a single assumed date.

---

## 10. Findings, Forecast, and Conclusion

In [ ]:
findings = {
    "Current premium (Feb 2026)":               "S$108,550 (avg of 2 rounds)",
    "Peak premium in sample":                   "S$129,611 (2023)",
    "Full-sample mean / median":                "S$50,131 / S$44,398",
    "Post-2016 mean / median":                  "S$67,773 / S$53,243  (+77% vs pre-2016 mean S$38,354)",
    "Bid-to-quota ratio (mean, range)":         "1.46x  (1.07x – 3.12x)",
    "Post-2016 excess demand (mean)":           "993 bids above quota per month",
    "Rolling volatility: pre / post-2016":      "σ=0.11 → 0.06 (log returns)",
    "ADF: quota_premium in levels":             "p=0.927 — unit root not rejected",
    "ADF: premium_return":                      "p=0.002 — stationary",
    "Granger: d_dereg → d_quota":               "p=0.003 at lag 6 — H1 confirmed",
    "Granger: d_quota → d_bids":                "p<0.001 at lag 5",
    "Granger: d_quota → premium_return":        "p=0.021 at lag 4",
    "Granger: d_new → premium_return":          "p=0.026 at lag 2",
    "Granger: d_bid_to_quota → premium_return": "p=0.798 — H2 not confirmed",
    "OLS: d_quota coefficient (HAC SE)":        "−0.0002  (p<0.001) — 1,000-unit quota rise ≈ −2% return",
    "OLS: d_bid_to_quota coefficient":          "+0.045  (p=0.415, not significant)",
    "OLS R²":                                   "0.077 — supply/auction variables explain ~8% of return variance",
    "Ljung-Box on residuals (lag 6)":           "p<0.001 — residual autocorrelation remains",
    "ARIMAX vs Naive (RMSE / MAE)":             "0.0536 / 0.0420 vs 0.0560 / 0.0452 — ARIMAX better on MAE",
}

print("=" * 72)
print("DATA-BACKED FINDINGS — COE Category A Premium Dynamics")
print("=" * 72)
for k, v in findings.items():
    print(f"\n  {k}:\n    {v}")

### 10.1 Hypothesis Assessment

| Hypothesis | Verdict | Evidence |
|---|---|---|
| **H1** — Deregistrations Granger-cause quota at 6–9 month lag | Confirmed | p=0.003 at lag 6; consistent with LTA's administrative processing cycle |
| **H2** — Bid-to-quota ratio dominates short-run premium movements | Rejected | Granger p=0.798; OLS p=0.415 (HAC). The ratio is an equilibrium outcome, not a forcing variable — it adjusts with price, not before it |
| **H3** — Levels correlations are spurious | Confirmed | ADF p=0.927 in levels; correlations collapse in differences; OLS R²=0.077 in first differences |
| **H4** — Regime dependence | Confirmed | Chow test confirms structural break; post-break mean +77%; volatility halved; rolling OLS shows coefficient shift. Break date validated by grid search |

The rejection of H2 is the most instructive finding. It does not mean demand pressure is irrelevant to COE premiums — it manifestly is. It means the bid-to-quota ratio, as a contemporaneous equilibrium measure, cannot predict where the market clears *next*. Quota changes, which are predetermined by past deregistrations before the auction opens, carry the predictive signal precisely because they are not set by the market.

### 10.2 Five-Year Forecast Framework (2026–2031)

The walk-forward evaluation confirms that all three models perform similarly on RMSE (~0.054–0.056). The naive random walk is competitive — a standard result for financial time series with low autocorrelation. The ARIMAX marginally outperforms on MAE but not RMSE. The conclusion is unambiguous: short-run premium returns are not reliably forecastable from public administrative data.

The 5-year scenario framework therefore abandons time-series extrapolation and is built instead on three structural drivers:

1. **Cohort deregistration wave.** Vehicles registered in 2016–2021, when annual Cat A registrations averaged ~25–30k/year, reach 10-year COE expiry between 2026 and 2031. This is mechanically predictable: those cars *will* deregister, those deregistrations *will* feed into quota at a 6-month lag (Granger p=0.003), and more quota *will* exert downward pressure on premium returns (OLS: −0.0002 per unit, p<0.001). The cohort wave is not a forecast — it is arithmetic.

2. **Demand regime.** The post-break regime is characterised by replacement buyers with inelastic demand. The question is whether that inelasticity is sufficient to absorb the incoming supply increase without a meaningful price correction, or whether even replacement demand softens as premiums remain elevated.

3. **Macro tail risks.** The current environment — Middle East instability, elevated oil prices, US tariff headwinds, MAS downward GDP revision — is asymmetrically skewed to the downside for premiums. A recession or sustained oil shock would amplify the bear scenario. Neither supports the bull scenario.

**Bear (supply relief + demand softening):** The cohort wave increases quota materially from 2026. Demand moderates as macro headwinds compress household discretionary income. Premium reverts toward ~S$48k by 2031 — roughly the post-break median.

**Base (cohort wave absorbed):** Quota rises as the cohort deregisters, but replacement demand remains resilient. The supply increase is partially absorbed. Premium drifts to ~S$82k by 2031, remaining elevated relative to the pre-break era but well below current levels.

**Bull (demand surge offsets supply):** PHV fleet renewal, population policy reversal, or credit easing generates demand that fully absorbs the cohort-driven quota increase. Premium holds near ~S$118k by 2031.

Note that even the bull scenario does not assume quota stays flat — it assumes demand rises fast enough to offset the mechanically rising supply. A truly flat-quota bull case is not consistent with the cohort data.

In [ ]:
np.random.seed(42)

lp = df["log_premium"].dropna()
sarimax_model = SARIMAX(lp, order=(1, 1, 1), trend="c").fit(disp=False)

n_ahead = 60
fc      = sarimax_model.get_forecast(steps=n_ahead)
fc_mean = np.exp(fc.predicted_mean)
fc_ci   = np.exp(fc.conf_int())

last_date = lp.index[-1]
fc_dates  = pd.date_range(last_date + pd.DateOffset(months=1), periods=n_ahead, freq="MS")

current_premium = 108550.0
post2016_sigma  = 0.06
post2016_median = 53243.0

scenarios = {
    "Bear (supply relief)": {"target": 48000,  "color": ACCENT3},
    "Base (status quo)":    {"target": 82000,  "color": ACCENT4},
    "Bull (demand surge)":  {"target": 118000, "color": ACCENT2},
}

ar1_phi = 0.35
scenario_paths = {}
for name, params in scenarios.items():
    log_trend = np.linspace(np.log(current_premium), np.log(params["target"]), n_ahead)
    noise     = np.zeros(n_ahead)
    eps       = np.random.normal(0, post2016_sigma, n_ahead)
    for t in range(1, n_ahead):
        noise[t] = ar1_phi * noise[t-1] + eps[t]
    scenario_paths[name] = np.exp(log_trend + noise * 0.5)

fig, ax = plt.subplots(figsize=(15, 7))
fig.suptitle("Figure 9 — COE Category A: 5-Year Scenario Forecast (2026–2031)", fontsize=13)

hist_plot = df["quota_premium"].dropna()
hist_plot = hist_plot[hist_plot.index >= "2015-01-01"]
ax.plot(hist_plot.index, hist_plot / 1000, color="#aaa", lw=1.5, label="Historical", zorder=3)

ax.plot(fc_dates, fc_mean / 1000, color=ACCENT, lw=1.5, ls="--",
        label="SARIMAX(1,1,1) point forecast", zorder=4)
ax.fill_between(fc_dates, fc_ci.iloc[:, 0] / 1000, fc_ci.iloc[:, 1] / 1000,
                color=ACCENT, alpha=0.10, label="95% CI")

for name, path in scenario_paths.items():
    c = scenarios[name]["color"]
    ax.plot(fc_dates, path / 1000, color=c, lw=2.0, label=name, zorder=5)
    ax.annotate(f"S${path[-1]/1000:.0f}k",
                xy=(fc_dates[-1], path[-1] / 1000),
                xytext=(8, 0), textcoords="offset points",
                fontsize=8, color=c, va="center")

ax.axhline(post2016_median / 1000, color="#555", lw=0.8, ls=":",
           label=f"Post-2016 median (S${post2016_median/1000:.0f}k)")
ax.axvline(pd.Timestamp("2026-02-01"), color="#555", lw=0.8, ls="--")
ax.text(pd.Timestamp("2026-03-01"), ax.get_ylim()[0] + 2, "Forecast start", fontsize=7, color="#888")
ax.axvspan(pd.Timestamp("2016-01-01"), pd.Timestamp("2026-02-01"), alpha=0.04, color=ACCENT3)

ax.set_ylabel("COE Premium (S$'000)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8, loc="upper left")
ax.set_xlim(pd.Timestamp("2015-01-01"), fc_dates[-1] + pd.DateOffset(months=3))

plt.tight_layout()
plt.show()

print("\n=== 5-Year Scenario Forecast Summary ===")
print(f"{'Scenario':<28} {'2026':<10} {'2027':<10} {'2028':<10} {'2029':<10} {'2030':<10} {'2031':<10}")
print("-" * 88)
for name, path in scenario_paths.items():
    annual = [f"S${np.mean(path[i*12:(i+1)*12])/1000:.0f}k" for i in range(5)]
    print(f"{name:<28} {annual[0]:<10} {annual[1]:<10} {annual[2]:<10} {annual[3]:<10} {annual[4]:<10} S${path[-1]/1000:.0f}k")

print(f"\nCurrent (Feb 2026): S${current_premium/1000:.0f}k  |  Post-2016 median: S${post2016_median/1000:.0f}k")
print(f"SARIMAX 5yr point:  S${fc_mean.iloc[-1]/1000:.0f}k  (wide CI reflects unit-root uncertainty)")

In [ ]:
reg_annual = (
    df["new_a"].dropna()
    .resample("YE").sum()
    .rename("new_registrations")
)
reg_annual.index = reg_annual.index.year

renewal_rate = 0.15  # ~15% renew at 10 years; remainder deregister

dereg_projected = {
    year: reg_annual[year - 10] * (1 - renewal_rate)
    for year in range(2026, 2032)
    if (year - 10) in reg_annual.index
}
dereg_proj_df = pd.Series(dereg_projected, name="projected_annual_dereg")

dereg_actual_annual = (
    df["dereg_a"].dropna()
    .resample("YE").sum()
    .rename("actual_annual_dereg")
)
dereg_actual_annual.index = dereg_actual_annual.index.year
recent_actual = dereg_actual_annual[dereg_actual_annual.index >= 2018]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 10 — Cohort Deregistration Wave: Structural Supply Driver", fontsize=12)

ax = axes[0]
reg_plot = reg_annual[(reg_annual.index >= 2002) & (reg_annual.index <= 2026)]
ax.bar(reg_plot.index, reg_plot / 1000, color=ACCENT, alpha=0.7)
for yr in range(2013, 2017):
    if yr in reg_plot.index:
        ax.bar(yr, reg_plot[yr] / 1000, color=ACCENT2, alpha=0.9)
ax.text(2014, reg_plot[2013:2017].max() / 1000 + 1,
        "2013–16 cohort\n→ deregisters 2023–26", fontsize=7, color=ACCENT2, ha="center")
ax.set_xlabel("Year of Registration")
ax.set_ylabel("Annual New Registrations ('000)")
ax.set_title("Cat A New Registrations by Year", fontsize=9)

ax = axes[1]
ax.bar(recent_actual.index, recent_actual / 1000,
       color=ACCENT, alpha=0.7, label="Actual")
ax.bar(dereg_proj_df.index, dereg_proj_df / 1000,
       color=ACCENT4, alpha=0.7, label="Projected (10yr cohort lag)")
ax.axvline(2025.5, color="#555", lw=0.8, ls="--")
ax.text(2025.6, ax.get_ylim()[1] * 0.95, "Forecast", fontsize=7, color="#888")
ax.set_xlabel("Year")
ax.set_ylabel("Annual Deregistrations ('000)")
ax.set_title("Actual vs Projected Cat A Deregistrations\n(85% deregistration assumption)", fontsize=9)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n=== Projected Annual Deregistrations (Cat A) ===")
for yr, val in dereg_proj_df.items():
    src = reg_annual.get(yr - 10, None)
    print(f"  {yr}: ~{val/1000:.1f}k  (from {yr-10} cohort: {src/1000:.1f}k registrations)"
          if src else f"  {yr}: ~{val/1000:.1f}k")

### 10.3 Forecast Interpretation and Macro Risk

The current premium of S$108,550 sits at +2.4 standard deviations above the post-break mean of S$67,773. In 24 years of data, premiums at this z-score have always corrected. The question is not whether they come down — the structural arithmetic makes that the base case — but how fast and how far.

**The cohort wave is the primary driver.** Figure 10 shows the 2016–2021 registration cohort reaching 10-year expiry between 2026 and 2031. Annual Cat A deregistrations are projected to rise from ~20k to ~28k over this window. At the Granger-confirmed 6-month transmission lag, this feeds directly into quota. The OLS coefficient implies each 1,000-unit monthly quota increase reduces premium returns by ~2%. Compounded over 5 years of rising quota, the directional pressure is unambiguous.

**The macro environment amplifies the downside.** The Israel-Iran escalation cycle has kept Brent crude volatile in the US$75–95/bbl range. A direct military exchange or Strait of Hormuz disruption — through which ~20% of global oil supply transits — would be a supply shock of a different order. Singapore imports all of its energy. A sustained oil price spike above US$120/bbl compresses household income and consumer confidence; the 2008 episode saw Cat A premiums fall 60% peak-to-trough as oil hit US$147/bbl. The US tariff escalation adds a second channel: Singapore's GDP is approximately 3.5× domestic consumption, and the economy is already absorbing the drag from semiconductor supply chain disruption and reduced export demand. MAS has revised its 2025 growth forecast downward. A recession — not the base case, but not a tail risk — would be the single most powerful downward force on premiums, dwarfing anything the supply mechanics alone can deliver.

These macro risks are not modelled — the data does not support it — but they are asymmetrically skewed to the downside. An oil shock or recession accelerates the bear scenario. Neither supports the bull.

---

### 10.4 The Singaporean Perspective

Let me be direct. Every Singaporean who has looked at a COE price in the last three years has had the same reaction: *this cannot be right*. Paying S$108,000 for the right to own a car — before the car itself — is not a market in equilibrium. It is a market under stress, and the data confirms it.

The bull case for sustained high premiums rests on structural demand having shifted permanently upward — more PHV drivers, wealthier households, a growing population. There is something to this. The post-break mean of S$67,773 is already 77% above the pre-break mean. The regime did shift. But S$108,550 is not the new normal. It is +2.4 standard deviations above even the elevated post-break mean. That is not a new equilibrium. That is an overshoot.

What brings it down is not a policy change or a market correction in the conventional sense. It is arithmetic. The cars bought in 2016–2021 are coming off the road. Their COEs are expiring. LTA will allocate more quota. More quota means lower premium returns. This chain is confirmed at every link by the data. The relief is already baked in — it just has not arrived yet.

The timing is the honest uncertainty. If the macro environment deteriorates — oil shock, recession, credit tightening — the correction arrives faster and goes deeper. If demand remains resilient and the cohort wave is absorbed, the base case of ~S$82k by 2031 is the landing zone. Either way, the direction is down from here.

For the Singaporean who has been waiting: the data is on your side. The supply mechanics are working in your favour. The macro headwinds are working in your favour. Patience, for once, is the analytically defensible strategy.

---

### 10.5 Conclusion

COE premiums are high because the VQS is designed to make them high when demand exceeds supply — and demand has exceeded supply for most of the past decade. That is not a market failure. It is the system doing exactly what it was built to do: price out marginal demand until the vehicle population stabilises.

What this analysis adds is precision about the mechanism and discipline about the forecast.

On mechanism: supply is not set by current demand. It is set by deregistrations 6 months ago. Quota changes are the dominant short-run predictor of premium returns. Level correlations are spurious. The post-break regime reflects a compositional shift in the buyer pool toward more inelastic replacement demand — which is why mean premiums are higher and volatility is lower. COVID-19 was a discrete shock that generated the most violent demand-supply dislocation in the series' history; its rebound is now substantially spent.

On forecast: the 5-year outlook is structurally bearish. The cohort deregistration wave is mechanically predictable and directionally unambiguous. The macro environment adds asymmetric downside risk that the model cannot quantify but cannot ignore. The base case of S$82k by 2031 is defensible under stable conditions. Under current conditions, the bear scenario deserves more probability weight than the scenario table implies.

The data says premiums should moderate. The cohort arithmetic says the same. The macro environment, for once, is also pointing in the same direction. The question is not whether COE premiums come down. It is whether they come down gradually or sharply.